# 🟢 12 — Multi-Turn Visual Chat: The Agent Remembers

> **Goal:** show that a Cosmos-Reason2 agent is a **conversation**, not a one-shot
> captioner. Ask three questions in a row about *one* image and watch the model carry
> context forward — turn 3 refers back to turn 1.

---

## Why this matters

Notebooks 01–05 each send a **single** prompt. But an agent's real power is *memory*:
Strands threads every earlier turn back in on each call, so the model reasons over the
whole conversation, not just the latest line. That is the difference between a captioner
and an assistant you can actually work with.

```mermaid
flowchart LR
    IMG["🖼 one image"]:::data --> A["🤖 one Agent<br/>(reused across turns)"]:::agent
    A --> T1["turn 1<br/>what objects?"]:::reason
    T1 --> T2["turn 2<br/>which to grasp first?"]:::reason
    T2 --> T3["turn 3<br/>recall turn 1"]:::reason
    T3 --> R([✅ answer references<br/>the earlier turns]):::out
    classDef agent fill:#E3F2FD,stroke:#1976D2,stroke-width:2px,color:#0D47A1
    classDef reason fill:#E8F5E9,stroke:#388E3C,stroke-width:2px,color:#1B5E20
    classDef data fill:#FFFDE7,stroke:#FBC02D,stroke-width:2px,color:#F57F17
    classDef out fill:#E0F2F1,stroke:#00897B,stroke-width:2px,color:#004D40
```

📄 Runnable script: [`../examples/12_multiturn_visual_chat.py`](../examples/12_multiturn_visual_chat.py)

In [ ]:
# 🔧 Preflight — this cell never crashes. It just tells us what's available.
import os, sys, time
os.environ.setdefault("QWEN_VL_VIDEO_READER", "torchcodec")
sys.path.insert(0, os.path.abspath(".."))

# 🔒 Cosmos tools confine file access to a workspace allow-list for safety. The
# notebooks live in notebooks/ but our sample media sits one level up, so we widen
# the workspace to the project root + /tmp. (Grant only what you need — no wider.)
os.environ["COSMOS_WORKSPACE"] = os.pathsep.join([os.path.abspath(".."), "/tmp"])

PROJECT_ROOT = os.path.abspath("..")
SAMPLE_IMAGE = os.environ.get("SAMPLE_IMAGE", os.path.join(PROJECT_ROOT, "sample.png"))

def gpu_ready() -> bool:
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False

READY = gpu_ready()
print("🎮 GPU available :", READY)
if READY:
    import torch
    print("   Device       :", torch.cuda.get_device_name(0))
    print(f"   VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("   (CPU-only is fine for reading along — the model cell will skip safely.)")
print("🖼  sample.png   :", os.path.exists(SAMPLE_IMAGE))

## Step 1 — Look at the image
The whole conversation is about this one picture (no GPU needed to display it).

In [ ]:
if os.path.exists(SAMPLE_IMAGE):
    from PIL import Image
    from IPython.display import display
    img = Image.open(SAMPLE_IMAGE)
    print(f"🖼  {img.size[0]}x{img.size[1]} pixels")
    display(img.resize((640, int(640 * img.size[1] / img.size[0]))))
else:
    print("⏭  sample.png not found — set SAMPLE_IMAGE to a real image.")

## Step 2 — Load the model once, reuse it

The key move: **one `Agent`, reused for every turn.** Strands keeps the conversation
history on that agent, so each new question is sent along with everything said before.

In [ ]:
agent = None
if READY and os.path.exists(SAMPLE_IMAGE):
    from strands import Agent
    from strands_cosmos import CosmosVisionModel
    t0 = time.time()
    model = CosmosVisionModel(
        model_id="nvidia/Cosmos-Reason2-2B",
        params={"max_tokens": 512, "temperature": 0.3},
    )
    agent = Agent(model=model)   # <-- reused across all turns below
    print(f"✅ Ready in {time.time()-t0:.1f}s")
else:
    print("⏭  Need a GPU + sample.png for the model turns.")

## Turn 1 — What's in the scene?
Note the **`<image>...</image>`** tag. We only attach the image once; later turns rely on
the agent remembering it.

In [ ]:
answers = []
if agent is not None:
    reply = str(agent(f"<image>{SAMPLE_IMAGE}</image> What objects are on the table?")).strip()
    answers.append(reply)
else:
    print("⏭  Skipped (no GPU).")

## Turn 2 — Reason about what to do
No image this time — just a follow-up question. The model still "sees" the scene because
it is in the conversation history.

In [ ]:
if agent is not None:
    reply = str(agent(
        "Which single object should a robot arm pick up first to start clearing the "
        "table? Name the object and give one reason."
    )).strip()
    answers.append(reply)
else:
    print("⏭  Skipped (no GPU).")

## Turn 3 — Prove it remembered
We ask the model to confirm its *first* answer. A "yes" that restates the object is the
observable signal that turns 1 and 2 were carried forward as context.

In [ ]:
if agent is not None:
    reply = str(agent("Was that object in your first answer? Reply yes or no and restate it.")).strip()
    answers.append(reply)
    kept_context = "yes" in reply.lower()
    print("\n" + "="*60)
    print(f"turns completed : {len(answers)}")
    print(f"context carried : {kept_context}  (turn 3 confirmed the earlier answer)")
    print("="*60)
else:
    print("⏭  Skipped (no GPU).")

# 🎓 What you learned

- **Reuse one `Agent`** across calls and Strands threads the prior turns back in — the
  model reasons over the whole conversation, not just the latest prompt.
- **Attach media once.** The `<image>` tag on turn 1 stays "visible" to later turns.
- **Verify, don't assume.** We checked that turn 3 recalled turn 1 rather than trusting a
  bare run — the same habit the [script](../examples/12_multiturn_visual_chat.py) uses
  (it exits non-zero if a turn returns nothing).

**Next:** [05 — Tool Usage](05_tool_usage.ipynb) to hand this model to another agent as a
tool, or [04 — Embodied Reasoning](04_embodied_reasoning.ipynb) for single-turn `<think>`
robot reasoning. 🚀